# 02 - Prototipagem e Impressao 3D
> G-code, otimizacao de geometria e documentacao tecnica com Claude

**Modulo:** `12_enfitec_servicos` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

## Otimizando Geometrias para Impressao

Claude pode analisar descricoes de pecas e sugerir modificacoes
para melhorar a imprimibilidade, reduzir suportes e aumentar a resistencia.

In [ ]:
system_3d = """Voce e um engenheiro especialista em Design for Additive Manufacturing (DfAM)
com experiencia em impressao FDM, SLA e SLS. Analise pecas e sugira
modificacoes para melhor imprimibilidade. Estruture sua resposta em:
PROBLEMAS IDENTIFICADOS, MODIFICACOES SUGERIDAS, ORIENTACAO DE IMPRESSAO
RECOMENDADA e PARAMETROS SUGERIDOS."""

peca_descricao = """Preciso imprimir em FDM (impressora Ender 3 V2) uma caixa para
abrigar o sensor de vibracao com estas caracteristicas:

- Dimensoes externas: 40x40x25mm
- Espessura de parede: 1.5mm
- Tampa com encaixe snap-fit (4 travas nas laterais)
- Furo passante de 8mm no fundo para passagem de cabo
- Abas de fixacao lateral com furos M4 (2x em cada lado)
- Canal interno para antena BLE (sulco de 2x1mm na tampa)
- Vedacao IP65 necessaria (anel o-ring na juncao tampa/base)
- Material: PETG

Quais problemas de impressao voce identifica e como otimizar o design?"""

resp = ask(peca_descricao, system=system_3d, model=SONNET, max_tokens=2048)
print(resp)

In [ ]:
# Detalhamento do snap-fit
snapfit = """Detalhe o design dos snap-fits para a caixa descrita acima.
Preciso de:
- Dimensoes recomendadas para as travas (comprimento, largura, angulo)
- Espessura da viga flexivel para PETG
- Deflexao maxima antes de quebra
- Forca de insercao/remocao estimada
- Numero minimo de ciclos de abertura/fechamento

Considere que PETG tem modulo de elasticidade ~2.0 GPa
e elongacao na ruptura ~23%. Forneca as formulas usadas."""

resp2 = ask(snapfit, system=system_3d, model=SONNET, max_tokens=2048)
print(resp2)

---
## Revisando e Gerando G-code

Claude pode interpretar, explicar e ate gerar trechos de G-code
para impressoras 3D e maquinas CNC.

In [ ]:
system_gcode = """Voce e um especialista em G-code para impressao 3D FDM.
Explique cada comando de forma clara e sugira otimizacoes.
Quando gerar G-code, inclua comentarios explicativos em cada linha."""

gcode_snippet = """Explique o que cada linha deste G-code faz e identifique
possiveis problemas ou otimizacoes:

G28 ; Home all axes
G29 ; Auto bed leveling
M104 S240 ; Set hotend temp
M140 S80 ; Set bed temp
M109 S240 ; Wait for hotend
M190 S80 ; Wait for bed
G92 E0 ; Reset extruder
G1 Z2.0 F3000 ; Move Z up
G1 X0.1 Y20 Z0.3 F5000.0 ; Move to start
G1 X0.1 Y200 Z0.3 F1500.0 E15 ; Draw 1st line
G1 X0.4 Y200 Z0.3 F5000.0 ; Move to side
G1 X0.4 Y20 Z0.3 F1500.0 E30 ; Draw 2nd line
G92 E0 ; Reset extruder
G1 Z1.0 F3000 ; Move Z up

Esse trecho e para PETG em uma Ender 3 V2. Algo errado?"""

resp3 = ask(gcode_snippet, system=system_gcode, model=SONNET, max_tokens=2048)
print(resp3)

In [ ]:
# Gerar G-code para forma simples
gerar_gcode = """Gere o G-code completo para imprimir um cilindro de teste
com estas especificacoes:

- Diametro: 20mm
- Altura: 10mm (5 camadas de 0.2mm cada... total 50 camadas)
- Paredes: 2 perimetros
- Sem preenchimento (vaso/vase mode nao, mas oco por dentro)
- Nozzle: 0.4mm
- Layer height: 0.2mm
- Velocidade de impressao: 40mm/s
- Material: PETG (hotend 235C, bed 80C)

Gere apenas as primeiras 3 camadas para demonstrar o padrao.
Inclua start G-code e comentarios explicativos."""

resp4 = ask(gerar_gcode, system=system_gcode, model=SONNET, max_tokens=3000)
print(resp4)

---
## Resistencia Mecanica vs Material

Comparacao de materiais para impressao 3D e uma tarefa onde Claude
pode gerar tabelas comparativas uteis para decisao de engenharia.

In [ ]:
system_materiais = """Voce e um engenheiro de materiais especialista em polimeros
para manufatura aditiva. Forneca dados tecnicos precisos e
recomendacoes praticas. Quando possivel, cite valores tipicos
de datasheets de fabricantes conhecidos."""

comparacao = """Compare os seguintes materiais para impressao FDM aplicados
a uma caixa de protecao para sensor industrial (ambiente: fabrica,
temperatura 0-60C, exposicao a oleos e vibracoes):

Materiais: PLA, ABS, PETG, Nylon (PA6/PA12)

Gere uma tabela comparativa com estas propriedades:
- Resistencia a tracao (MPa)
- Modulo de elasticidade (GPa)
- Resistencia ao impacto (kJ/m2)
- Temperatura de deflexao termica - HDT (C)
- Resistencia quimica (oleo mineral, oleo lubrificante)
- Absorcao de umidade (%)
- Facilidade de impressao (1-5)
- Custo relativo (R$/kg tipico)
- Necessidade de enclosed chamber
- Shrinkage/warping (baixo/medio/alto)

Ao final, recomende o melhor material para esta aplicacao com justificativa."""

resp5 = ask(comparacao, system=system_materiais, model=SONNET, max_tokens=2048)
print(resp5)

In [ ]:
# Parametros de impressao otimizados por material
parametros = """Para o material PETG (que voce recomendou), forneca os parametros
otimizados de impressao para a Ender 3 V2 com upgrade de direct drive:

Retorne como dicionario Python:
params_petg = {
    'temperatura_nozzle_c': ...,
    'temperatura_bed_c': ...,
    'velocidade_impressao_mm_s': ...,
    'velocidade_primeira_camada_mm_s': ...,
    'layer_height_mm': ...,
    'primeira_camada_height_mm': ...,
    'retraction_distance_mm': ...,
    'retraction_speed_mm_s': ...,
    'fan_speed_percent': ...,
    'fan_primeira_camada_percent': ...,
    'infill_percent': ...,
    'wall_count': ...,
    'top_bottom_layers': ...,
    'suporte_angulo_graus': ...,
    'z_hop_mm': ...
}

Inclua comentarios explicando o motivo de cada valor."""

resp6 = ask(parametros, system=system_materiais, model=HAIKU, max_tokens=2048)
print(resp6)

---
## Tolerancias e Ajuste de Pecas

Um dos maiores desafios da impressao 3D e acertar as tolerancias
para encaixes mecanicos. Claude pode ajudar com recomendacoes praticas.

In [ ]:
system_tolerancia = """Voce e um projetista mecanico especialista em tolerancias
dimensionais para pecas impressas em 3D (FDM). Forneca valores
praticos baseados em experiencia real, nao apenas teoria.
Considere as limitacoes tipicas de impressoras desktop."""

tolerancias = """Preciso projetar os seguintes encaixes para pecas impressas
em PETG na Ender 3 V2 (resolucao pratica ~0.1mm):

1. PRESS-FIT: Eixo cilindrico macho/femea para fixacao permanente
   - Diametro nominal: 8mm
   - Qual interferencia usar?

2. SNAP-FIT: Trava elastica para tampa removivel
   - Qual gap/folga entre macho e femea?
   - Angulo de entrada vs angulo de retencao?

3. SLIDING FIT: Guia linear para peca deslizante
   - Secao retangular 10x5mm
   - Qual folga para movimento suave sem jogo excessivo?

4. THREADED INSERT: Furo para inserto termico M3
   - Diametro do furo recomendado?
   - Profundidade minima?

5. O-RING GROOVE: Canal para anel o-ring (vedacao IP65)
   - O-ring cord diameter: 1.5mm
   - Dimensoes do canal (largura x profundidade)?

Para cada caso, forneca:
- Dimensoes nominais e tolerancias recomendadas
- Compensacao para shrinkage do PETG
- Orientacao de impressao preferencial
- Dicas praticas de pos-processamento"""

resp7 = ask(tolerancias, system=system_tolerancia, model=SONNET, max_tokens=3000)
print(resp7)

In [ ]:
# Tabela de compensacao de tolerancias
tabela_tol = """Gere uma tabela de referencia rapida de compensacao de tolerancias
para impressao FDM em PETG. Formate como dicionario Python:

tolerancias_ref = {
    'tipo_encaixe': ['press_fit', 'snap_fit', 'sliding', 'clearance',
                     'threaded_insert_m3', 'threaded_insert_m4',
                     'oring_1.5mm', 'oring_2mm'],
    'folga_diametral_mm': [...],
    'folga_linear_mm': [...],
    'orientacao_preferencial': [...],
    'pos_processamento': [...]
}

Esses valores devem funcionar como ponto de partida confiavel
para a maioria dos projetos em PETG."""

resp8 = ask(tabela_tol, system=system_tolerancia, model=HAIKU, max_tokens=2048)
print(resp8)

---
## Especificacao Tecnica Automatica

A partir de um briefing de projeto, Claude pode gerar uma especificacao
tecnica completa para o prototipo, pronta para revisao.

In [ ]:
system_spec = """Voce e um engenheiro de produto que cria especificacoes tecnicas
detalhadas para prototipos. Use formato profissional com numeracao
de secoes, tabelas e requisitos rastreiaveis (REQ-XXX). O documento
deve ser completo o suficiente para um projetista CAD iniciar o trabalho."""

briefing = """Gere a ESPECIFICACAO TECNICA DO PROTOTIPO para:

PROJETO: Caixa de Protecao para Sensor de Vibracao Wireless
CODIGO: ENF-SV-001-PROTO
REVISAO: A (primeiro prototipo)

BRIEFING DO PROJETO:
Caixa para abrigar placa eletronica de sensor de vibracao (PCB 25x25mm),
bateria Li-SOCl2 (tamanho AA), antena BLE chip e conector de cabo.
Deve ser montada com fixacao magnetica ou parafusos M4 na carcaca
de motores eletricos industriais. Ambiente: fabrica, temperatura
0-60C, exposicao a poeira e respingos de oleo.

REQUISITOS CHAVE:
- Protecao IP65
- Resistente a vibracoes (montada em motor)
- Facil abertura para troca de bateria (sem ferramentas)
- Material: PETG (prototipo) / PA6-GF30 (producao)
- Fabricacao: impressao 3D FDM (prototipo) / injecao (producao)
- Fixacao: imas neodimio N42 (2x 10x3mm) + opcao parafuso M4

A especificacao deve conter:
1. Escopo e objetivo
2. Documentos de referencia (normas IP, IEC, etc.)
3. Requisitos mecanicos (dimensoes, tolerancias, resistencia)
4. Requisitos ambientais (temperatura, IP, vibracao)
5. Requisitos de montagem (snap-fit, o-ring, imas)
6. Bill of Materials mecanico
7. Criterios de aceitacao do prototipo
8. Testes de validacao requeridos"""

spec = ask(briefing, system=system_spec, model=SONNET, max_tokens=4096)
print(spec)

In [ ]:
# Salvar especificacao em arquivo
with open('spec_ENF-SV-001-PROTO_revA.md', 'w', encoding='utf-8') as f:
    f.write(spec)

print('Especificacao salva em: spec_ENF-SV-001-PROTO_revA.md')
print(f'Tamanho: {len(spec)} caracteres')

In [ ]:
# Gerar checklist de revisao do prototipo
checklist = """Com base na especificacao acima, gere um CHECKLIST DE REVISAO
do prototipo impresso em 3D. O checklist deve ser pratico, com itens
verificaveis tipo "passa/nao passa".

Categorias:
1. Dimensional (tolerancias criticas)
2. Montagem (snap-fit funciona? o-ring assenta?)
3. Funcional (PCB encaixa? bateria cabe? cabo passa?)
4. Ambiental (teste de respingo, teste de poeira basico)
5. Fixacao (imas seguram? parafusos alinham?)

Formate como lista markdown com checkbox [ ]."""

resp9 = ask(checklist, system=system_spec, model=HAIKU, max_tokens=2048)
print(resp9)

---
## Exercicios

1. **Outra Peca**: Descreva uma peca diferente (suporte de camera, case de
   Raspberry Pi, engrenagem) e peca a Claude para otimizar para FDM.

2. **G-code Avancado**: Peca a Claude para gerar G-code de uma purge tower
   ou de um teste de retracao (retraction tower) parametrico.

3. **Material Exotico**: Pesquise com Claude sobre impressao em materiais
   como TPU, PC (policarbonato) ou ASA. Quando usar cada um?

4. **Custo de Impressao**: Peca a Claude para calcular o custo de impressao
   (material + eletricidade + depreciacao da maquina) de uma peca.

5. **Transicao para Injecao**: Peca a Claude para listar as modificacoes
   de design necessarias ao migrar uma peca de impressao 3D para
   moldagem por injecao (draft angles, wall thickness, etc.).

---
## Proximos Passos

- **Notebook 03**: Firmware assistido por IA - desenvolvimento de codigo
  embarcado para ESP32 com auxilio de Claude.
- **Notebook 04**: Documentacao tecnica automatizada - datasheets, manuais
  e especificacoes completas gerados com IA.
- **Notebook 05**: Testes e validacao - criacao de planos de teste
  e analise de resultados com Claude.

---
*Notebook criado como parte do modulo 12_enfitec_servicos*